In [1]:
import sys
print(sys.executable)

c:\Users\Admin\anaconda3\envs\tf_env\python.exe


In [2]:
!pip install scikeras

In [3]:
import scikeras
print(scikeras.__version__)


0.12.0


In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [5]:
data=pd.read_csv('Churn_Modelling.csv')
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo=OneHotEncoder()
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df= pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

X=data.drop('Exited',axis=1)
y=data['Exited']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

## Save the encoder and scaler and scaler for later use
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl','wb')as file:
    pickle.dump(onehot_encoder_geo,file)

with open('scaler.pkl','wb')as file:
    pickle.dump(scaler,file)

In [6]:
## Define a function to create a model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [7]:
## Create a Keras Classifier
model=KerasClassifier(layers=1, neurons=32, build_fn=create_model,verbose=1)

In [8]:
## Define Grid Search Parameters
param_grid={
    'neurons':[16,32,64,128],
    'layers':[1,2],
    'epochs':[50,100]
}

In [9]:
## Perform Grid Search 
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)

## Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


c:\Users\Admin\anaconda3\envs\tf_env\lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)




Epoch 1/100


250/250 [==============================] - 1s 2ms/step - loss: 0.4984 - accuracy: 0.7857
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 0.4400 - accuracy: 0.8101
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 0.4238 - accuracy: 0.8176
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 0.4119 - accuracy: 0.8240
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.4006 - accuracy: 0.8319
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3892 - accuracy: 0.8382
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3785 - accuracy: 0.8444
Epoch 8/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3698 - accuracy: 0.8466
Epoch 9/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3633 - accuracy: 0.8506
Epoch 10/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3582 - ac